<a href="https://colab.research.google.com/github/danielpazrosseboe/MasterDRC/blob/main/TFRecords_Band_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================
# DHS-style TFRecord band analysis (Yeh et al replication)
# ======================
!pip -q install tqdm tensorflow numpy

import os, glob, time
import numpy as np
import tensorflow as tf
from tqdm.auto import tqdm

# --- CONFIG ---
TFRECORD_DIR = "/content/drive/My Drive/dhs_tfrecords_raw"
BANDS = ['BLUE','GREEN','RED','SWIR1','SWIR2','TEMP1','NIR','NIGHTLIGHTS']
PATCH_SIZE = 255  # 255x255 patches → 65025 values
PIXELS_PER_IMAGE = PATCH_SIZE**2
K_TOP = 20  # to track k worst images (fewest good pixels)

# --- HELPER FUNCS ---
def parse_example(raw):
    ex = tf.train.Example.FromString(raw)
    f = ex.features.feature
    return f

def get_arrays(f):
    out = {}
    for b in BANDS:
        out[b] = np.asarray(f[b].float_list.value, dtype=np.float32) if b in f else np.zeros(PATCH_SIZE**2, np.float32)
    return out

def update_stats(stats, arrays, good_mask):
    for i, b in enumerate(BANDS):
        arr = arrays[b]
        if arr.size == 0: continue
        stats["mins"][i] = min(stats["mins"][i], arr.min())
        stats["maxs"][i] = max(stats["maxs"][i], arr.max())
        stats["sums"][i] += arr.sum(dtype=np.float64)
        stats["sum_sqs"][i] += np.square(arr, dtype=np.float64).sum()
        nz = np.count_nonzero(arr > 0)
        stats["nz_pixels"][i] += nz
        if nz > 0:
            stats["mins_nz"][i] = min(stats["mins_nz"][i], arr[arr > 0].min())
        stats["mins_goodpx"][i] = min(stats["mins_goodpx"][i], arr[good_mask].min() if np.any(good_mask) else np.inf)
    return stats

# --- INIT STATS ---
nbands = len(BANDS)
stats = {
    "mins": np.full(nbands, np.inf),
    "mins_nz": np.full(nbands, np.inf),
    "mins_goodpx": np.full(nbands, np.inf),
    "maxs": np.full(nbands, -np.inf),
    "sums": np.zeros(nbands, np.float64),
    "sum_sqs": np.zeros(nbands, np.float64),
    "nz_pixels": np.zeros(nbands, np.int64),
}
num_good_pixels = []

# --- FILE LIST ---
paths = sorted(glob.glob(os.path.join(TFRECORD_DIR, "*.tfrecord.gz")))
if not paths:
    raise SystemExit(f"No .tfrecord.gz files in {TFRECORD_DIR}")
print(f"Found {len(paths)} TFRecords. Starting full-band analysis…")

# --- MAIN LOOP ---
start = time.time()
for path in tqdm(paths, desc="Processing TFRecords"):
    try:
        ds = tf.data.TFRecordDataset(path, compression_type="GZIP")
        for raw in ds.as_numpy_iterator():
            f = parse_example(raw)
            arrays = get_arrays(f)
            # good pixel mask = any band > 0 (excluding negatives)
            good_mask = np.zeros(PATCH_SIZE**2, dtype=bool)
            for b in BANDS:
                good_mask |= (arrays[b] > 0)
            stats = update_stats(stats, arrays, good_mask)
            num_good_pixels.append(int(good_mask.sum()))
    except Exception as e:
        print(f"Error reading {os.path.basename(path)}: {e}")

elapsed = time.time() - start
images_count = len(num_good_pixels)
total_pixels = images_count * PIXELS_PER_IMAGE
print(f"\nProcessed {images_count} images in {elapsed:.1f}s")

# --- SUMMARY ---
def print_summary(stats, num_good_pixels):
    total_good = np.sum(num_good_pixels)
    print("\n=== Statistics including bad pixels ===")
    means = stats["sums"]/total_pixels
    stds = np.sqrt(stats["sum_sqs"]/total_pixels - means**2)
    for i,b in enumerate(BANDS):
        print(f"{b:8s} mean={means[i]:10.6f}, std={stds[i]:10.6f}, min={stats['mins'][i]:10.6g}, max={stats['maxs'][i]:10.6g}")

    print("\n=== Statistics ignoring ≤0 values ===")
    nz = np.maximum(stats["nz_pixels"], 1)
    means_nz = stats["sums"]/nz
    stds_nz = np.sqrt(np.maximum(0, stats["sum_sqs"]/nz - means_nz**2))
    avg_nz = stats["nz_pixels"].astype(np.float32)/images_count
    for i,b in enumerate(BANDS):
        print(f"{b:8s} mean={means_nz[i]:10.6f}, std={stds_nz[i]:10.6f}, min_nz={stats['mins_nz'][i]:10.6g}, "
              f"max={stats['maxs'][i]:10.6g}, mean_nz={avg_nz[i]:.1f}")

    print("\n=== Statistics excluding bad pixels (union mask) ===")
    means_good = stats["sums"]/total_good
    stds_good  = np.sqrt(np.maximum(0, stats["sum_sqs"]/total_good - means_good**2))
    for i,b in enumerate(BANDS):
        print(f"{b:8s} mean={means_good[i]:10.6f}, std={stds_good[i]:10.6f}, min_good={stats['mins_goodpx'][i]:10.6g}, "
              f"max={stats['maxs'][i]:10.6g}")
    print(f"\nAverage good pixels per image: {np.mean(num_good_pixels):.1f} / {PIXELS_PER_IMAGE}")
    print(f"Min/Max good pixels per image: {np.min(num_good_pixels)} / {np.max(num_good_pixels)}")

print_summary(stats, num_good_pixels)
